In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
import statsmodels.api as sm
import pymc as pm
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score
from sklearn.preprocessing import MinMaxScaler

# ==========================================
# 1. PURE DATA ENGINE (48 Variables)
# ==========================================
class PureMMMDataEngine:
    def __init__(self, target_col='units_sold', date_col='week_start_date'):
        self.target_col = target_col
        self.date_col = date_col
        self.scaler = MinMaxScaler()

    def process_data(self, filepath):
        print("🚀 INITIALIZING PURE DATA ENGINE (48 Variables)...")
        df = pd.read_csv(filepath)

        if self.date_col in df.columns:
            df[self.date_col] = pd.to_datetime(df[self.date_col])
            df = df.sort_values(self.date_col).reset_index(drop=True)

        cols_to_drop = [self.target_col, self.date_col]
        X = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
        y = df[self.target_col]

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

        X_train_scaled = pd.DataFrame(self.scaler.fit_transform(X_train), columns=X_train.columns)
        X_test_scaled = pd.DataFrame(self.scaler.transform(X_test), columns=X_test.columns)

        print(f"📊 Splitting Complete: {X_train_scaled.shape[1]} pure features ready for training.\n")
        return X_train_scaled, X_test_scaled, y_train, y_test

# ==========================================
# 2. MASTER UNIFIED TRACKER
# ==========================================
class UnifiedMMMTracker:
    def __init__(self, X_train, y_train, X_test, y_test):
        self.X_train = X_train
        self.y_train = y_train
        self.X_test = X_test
        self.y_test = y_test
        self.trained_models = {}
        self.predictions = {}
        self.feature_weights = {}

    def train_linear_models(self):
        lr = LinearRegression()
        lr.fit(self.X_train, self.y_train)
        self.trained_models['Linear Regression'] = lr
        self.predictions['Linear Regression'] = lr.predict(self.X_test)
        self.feature_weights['Linear Regression'] = lr.coef_
        print("✅ Linear Regression model finished training.")

        ridge = Ridge(alpha=50.0)
        ridge.fit(self.X_train, self.y_train)
        self.trained_models['Ridge (L2 - Smooth Distribution)'] = ridge
        self.predictions['Ridge (L2 - Smooth Distribution)'] = ridge.predict(self.X_test)
        self.feature_weights['Ridge (L2 - Smooth Distribution)'] = ridge.coef_
        print("✅ Ridge (L2 - Smooth Distribution) finished training.")

        lasso = Lasso(alpha=2.0, max_iter=10000)
        lasso.fit(self.X_train, self.y_train)
        self.trained_models['Lasso (L1 - Strict Selection)'] = lasso
        self.predictions['Lasso (L1 - Strict Selection)'] = lasso.predict(self.X_test)
        self.feature_weights['Lasso (L1 - Strict Selection)'] = lasso.coef_
        print("✅ Lasso (L1 - Strict Selection) finished training.")

    def train_ols_model(self):
        X_train_sm = sm.add_constant(self.X_train)
        X_test_sm = sm.add_constant(self.X_test)
        ols_model = sm.OLS(self.y_train.values, X_train_sm).fit()
        self.trained_models['OLS Model'] = ols_model
        self.predictions['OLS Model'] = ols_model.predict(X_test_sm)
        self.feature_weights['OLS Model'] = ols_model.params[1:]
        print("✅ OLS Model finished training.")

    def train_tree_models(self):
        rf = RandomForestRegressor(n_estimators=300, max_depth=10, random_state=42)
        rf.fit(self.X_train, self.y_train)
        self.trained_models['Random Forest (Bagging)'] = rf
        self.predictions['Random Forest (Bagging)'] = rf.predict(self.X_test)
        self.feature_weights['Random Forest (Bagging)'] = rf.feature_importances_
        print("✅ Random Forest (Bagging) finished training.")

        xgb = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=5, random_state=42)
        xgb.fit(self.X_train, self.y_train)
        self.trained_models['XGBoost (Boosting)'] = xgb
        self.predictions['XGBoost (Boosting)'] = xgb.predict(self.X_test)
        self.feature_weights['XGBoost (Boosting)'] = xgb.feature_importances_
        print("✅ XGBoost (Boosting) finished training.")

    def train_bayesian_model(self, media_keywords=['tv', 'youtube', 'facebook', 'radio', 'tiktok', 'influencer']):
        # Dynamically separate features
        media_cols = [col for col in self.X_train.columns if any(k in col.lower() for k in media_keywords)]
        base_cols = [col for col in self.X_train.columns if col not in media_cols]

        with pm.Model() as bayes_model:
            # Mutable Data for predicting the future safely
            media_data = pm.Data('media_data', self.X_train[media_cols].values)
            base_data = pm.Data('base_data', self.X_train[base_cols].values)
            y_data = pm.Data('y_data', self.y_train.values)

            y_std = self.y_train.std()

            intercept = pm.Normal('intercept', mu=self.y_train.mean(), sigma=y_std)
            beta_base = pm.Normal('beta_base', mu=0, sigma=y_std, shape=len(base_cols))
            base_effect = pm.math.dot(base_data, beta_base)

            beta_media = pm.HalfNormal('beta_media', sigma=y_std, shape=len(media_cols))
            media_effect = pm.math.dot(media_data, beta_media)

            mu = intercept + base_effect + media_effect
            sigma = pm.HalfNormal('sigma', sigma=y_std)

            pm.Normal('units_sold', mu=mu, sigma=sigma, observed=y_data)

            trace = pm.sample(draws=1000, tune=1000, progressbar=False, return_inferencedata=True)

        self.trained_models['Bayesian (PyMC)'] = bayes_model

        # Predict on Test Data safely
        with bayes_model:
            pm.set_data({
                'media_data': self.X_test[media_cols].values,
                'base_data': self.X_test[base_cols].values,
                'y_data': self.y_test.values
            })
            post_pred = pm.sample_posterior_predictive(trace, progressbar=False)
            preds = post_pred.posterior_predictive['units_sold'].median(dim=['chain', 'draw']).values

        self.predictions['Bayesian (PyMC)'] = preds

        # Stitch weights back into original X_train column order so the master extraction table doesn't break
        full_weights = np.zeros(len(self.X_train.columns))
        median_beta_media = trace.posterior['beta_media'].median(dim=['chain', 'draw']).values
        median_beta_base = trace.posterior['beta_base'].median(dim=['chain', 'draw']).values

        for i, col in enumerate(self.X_train.columns):
            if col in media_cols:
                full_weights[i] = median_beta_media[media_cols.index(col)]
            else:
                full_weights[i] = median_beta_base[base_cols.index(col)]

        self.feature_weights['Bayesian (PyMC)'] = full_weights
        print("✅ Bayesian model finished training.")

    def generate_leaderboard(self):
        results = []
        for name, preds in self.predictions.items():
            mape = mean_absolute_percentage_error(self.y_test, preds) * 100
            rmse = np.sqrt(mean_squared_error(self.y_test, preds))
            r2 = r2_score(self.y_test, preds) # Calculate R-squared

            results.append({
                'Model Architecture': name,
                'MAPE (%)': round(mape, 2),
                'RMSE': round(rmse, 2),
                'R²': round(r2, 4) # Add it to the table
            })

        leaderboard = pd.DataFrame(results).sort_values(by='MAPE (%)', ascending=True).reset_index(drop=True)

        print("\n==============================================")
        print("🏆 Leaderboard")
        print("==============================================")
        print(leaderboard.to_string())
        return leaderboard

    def extract_contributions(self):
        print("\n==============================================")
        print("📊 MATHEMATICAL CONTRIBUTIONS (Cross-Model)")
        print("==============================================\n")
        categories = {
            'TV Spend': 'tv', 'Facebook Spend': 'facebook', 'YouTube Spend': 'youtube',
            'Radio Spend': 'radio', 'TikTok Spend': 'tiktok', 'Influencer Spend': 'influencer'
        }
        master_table = {}
        for model_name, weights in self.feature_weights.items():
            bucketed_pct = {k: 0.0 for k in categories.keys()}
            base_pct = 0.0

            if 'Forest' in model_name or 'XGBoost' in model_name:
                impacts = weights * 100
            else:
                raw_impacts = np.abs(weights * self.X_train.mean().values)
                total_impact = np.sum(raw_impacts)
                impacts = (raw_impacts / total_impact) * 100 if total_impact > 0 else raw_impacts

            for col_name, pct in zip(self.X_train.columns, impacts):
                matched = False
                for bucket_name, keyword in categories.items():
                    if keyword in col_name.lower():
                        bucketed_pct[bucket_name] += pct
                        matched = True
                        break
                if not matched: base_pct += pct

            col_data = [f"{base_pct:.2f}%"] + [f"{bucketed_pct[k]:.2f}%" for k in categories.keys()]
            master_table[model_name] = col_data

        index_labels = ['Base Sales (Macro/Dist)'] + list(categories.keys())
        df_contributions = pd.DataFrame(master_table, index=index_labels)
        print(df_contributions.to_string())
        return df_contributions

# --- EXECUTION BLOCK ---
data_engine = PureMMMDataEngine()
X_train, X_test, y_train, y_test = data_engine.process_data('fmcg_synthetic_mmm_dataset_v2.csv')

tracker = UnifiedMMMTracker(X_train, y_train, X_test, y_test)
tracker.train_linear_models()
tracker.train_tree_models()
tracker.train_ols_model()
tracker.train_bayesian_model()

leaderboard_df = tracker.generate_leaderboard()
contribution_df = tracker.extract_contributions()

🚀 INITIALIZING PURE DATA ENGINE (48 Variables)...
📊 Splitting Complete: 48 pure features ready for training.

✅ Linear Regression model finished training.
✅ Ridge (L2 - Smooth Distribution) finished training.
✅ Lasso (L1 - Strict Selection) finished training.
✅ Random Forest (Bagging) finished training.
✅ XGBoost (Boosting) finished training.
✅ OLS Model finished training.


ERROR:pymc.stats.convergence:There were 41 divergences after tuning. Increase `target_accept` or reparameterize.


✅ Bayesian model finished training.

🏆 Leaderboard
                 Model Architecture  MAPE (%)     RMSE      R²
0     Lasso (L1 - Strict Selection)      3.07   656.30  0.9726
1                   Bayesian (PyMC)      3.19   718.08  0.9671
2                         OLS Model      3.21   705.72  0.9683
3                 Linear Regression      3.21   705.72  0.9683
4                XGBoost (Boosting)      7.16  1569.76  0.8430
5           Random Forest (Bagging)      9.25  1962.58  0.7546
6  Ridge (L2 - Smooth Distribution)     14.29  3079.19  0.3959

📊 MATHEMATICAL CONTRIBUTIONS (Cross-Model)

                        Linear Regression Ridge (L2 - Smooth Distribution) Lasso (L1 - Strict Selection) Random Forest (Bagging) XGBoost (Boosting) OLS Model Bayesian (PyMC)
Base Sales (Macro/Dist)            49.96%                           61.04%                        47.03%                  36.65%             40.42%    49.96%          47.04%
TV Spend                            0.21%           

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score
import statsmodels.api as sm
import pymc as pm
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import MinMaxScaler
from itertools import product

# ==========================================
# 0. ADSTOCK + HILL SATURATION ENGINE
# ==========================================

class AdstockHillEngine:
    """
    Applies Geometric Adstock + Hill Saturation transformations
    to all media spend columns before MMM modeling.

    Why Adstock?
        Advertising builds memory in consumers that persists beyond
        the week it aired. Broadbent (1979): A(t) = E(t) + λ × A(t-1)

    Why Hill Saturation?
        Spending more eventually produces diminishing returns.
        Jin et al. (2017): H(x) = x^α / (x^α + γ^α)

    Order: Adstock FIRST → Hill Saturation SECOND
        Saturation captures fatigue from accumulated exposure,
        not just this week's spend.
    """

    # Candidate values to search over per channel
    LAMBDA_GRID = [0.3, 0.6, 0.9]   # decay rates
    ALPHA_GRID  = [1.5, 2.5]         # Hill slope
    GAMMA       = 0.5                 # half-saturation point (fixed)

    MEDIA_KEYWORDS = ['tv', 'facebook', 'youtube', 'radio', 'tiktok', 'influencer']

    def __init__(self):
        self.best_params   = {}   # {col: {'lambda': x, 'alpha': x}}
        self.transformed_cols = {}  # {col: best transformed series name}

    # ── Geometric Adstock ───────────────────────────────────────────────────
    @staticmethod
    def geometric_adstock(series: np.ndarray, lam: float) -> np.ndarray:
        """
        A(t) = E(t) + λ × A(t-1)
        λ = 0  → no carryover (each week independent)
        λ = 0.9 → strong carryover (effect lingers weeks)
        """
        adstocked = np.zeros_like(series, dtype=float)
        adstocked[0] = series[0]
        for t in range(1, len(series)):
            adstocked[t] = series[t] + lam * adstocked[t - 1]
        return adstocked

    # ── Hill Saturation ─────────────────────────────────────────────────────
    @staticmethod
    def hill_saturation(series: np.ndarray, alpha: float, gamma: float = 0.5) -> np.ndarray:
        """
        H(x) = x^α / (x^α + γ^α)
        α controls how steeply curve rises (slope)
        γ is the half-saturation point (spend level = 50% max response)
        Output is always between 0 and 1.
        """
        # Normalize to [0,1] first so Hill works correctly across channels
        s_min, s_max = series.min(), series.max()
        if s_max == s_min:
            return np.zeros_like(series)
        x = (series - s_min) / (s_max - s_min)
        return (x ** alpha) / (x ** alpha + gamma ** alpha)

    # ── Best param selection via correlation with target ─────────────────────
    def _best_params_for_col(self, col_series: np.ndarray, y: np.ndarray):
        """
        Grid search over (lambda, alpha) combinations.
        Pick the pair whose transformed series has highest
        absolute Pearson correlation with the sales target.
        This is the standard practitioner approach (Jin et al., 2017).
        """
        best_corr   = -1
        best_lam    = self.LAMBDA_GRID[1]
        best_alpha  = self.ALPHA_GRID[0]

        for lam, alpha in product(self.LAMBDA_GRID, self.ALPHA_GRID):
            ads  = self.geometric_adstock(col_series, lam)
            hill = self.hill_saturation(ads, alpha, self.GAMMA)
            corr = abs(np.corrcoef(hill, y)[0, 1])
            if np.isfinite(corr) and corr > best_corr:
                best_corr  = corr
                best_lam   = lam
                best_alpha = alpha

        return best_lam, best_alpha

    # ── Main transform method ────────────────────────────────────────────────
    def fit_transform(self, df: pd.DataFrame, y: pd.Series) -> pd.DataFrame:
        """
        For every media column:
          1. Grid search best (λ, α) using correlation with y
          2. Apply adstock then Hill
          3. Replace original column with transformed version
          4. Keep non-media columns unchanged
        Returns transformed DataFrame.
        """
        df_out = df.copy()
        media_cols = [c for c in df.columns
                      if any(k in c.lower() for k in self.MEDIA_KEYWORDS)]

        print("\n" + "="*60)
        print("📡 ADSTOCK + HILL SATURATION TRANSFORMATION")
        print("="*60)
        print(f"{'Channel':<35} {'Best λ':>8} {'Best α':>8} {'Corr with Sales':>16}")
        print("-"*60)

        for col in media_cols:
            raw = df[col].values.astype(float)
            lam, alpha = self._best_params_for_col(raw, y.values)
            self.best_params[col] = {'lambda': lam, 'alpha': alpha}

            # Apply transformation
            ads  = self.geometric_adstock(raw, lam)
            hill = self.hill_saturation(ads, alpha, self.GAMMA)

            # Correlation with target for reporting
            corr = np.corrcoef(hill, y.values)[0, 1]
            print(f"{col:<35} {lam:>8.1f} {alpha:>8.1f} {corr:>16.4f}")

            # Replace in output DataFrame
            transformed_name = f"{col}_adstock_hill"
            df_out[transformed_name] = hill
            df_out.drop(columns=[col], inplace=True)
            self.transformed_cols[col] = transformed_name

        print("="*60)
        print(f"✅ {len(media_cols)} media channels transformed.")
        print(f"   Order: Geometric Adstock → Hill Saturation (γ=0.5)")
        print(f"   Param selection: max correlation with units_sold\n")
        return df_out

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Apply SAME params learned during fit_transform to new data (test set).
        Critical: never refit on test data — that would be data leakage.
        """
        df_out = df.copy()
        for col, params in self.best_params.items():
            if col not in df.columns:
                continue
            raw  = df[col].values.astype(float)
            ads  = self.geometric_adstock(raw, params['lambda'])
            hill = self.hill_saturation(ads, params['alpha'], self.GAMMA)
            transformed_name = self.transformed_cols[col]
            df_out[transformed_name] = hill
            df_out.drop(columns=[col], inplace=True)
        return df_out

    def get_params_summary(self) -> pd.DataFrame:
        """Returns a clean summary table of best params per channel."""
        rows = []
        for col, p in self.best_params.items():
            rows.append({
                'Media Channel': col,
                'Best Lambda (λ)': p['lambda'],
                'Half-life (weeks)': round(np.log(0.5) / np.log(p['lambda']), 1) if p['lambda'] > 0 else '∞',
                'Best Alpha (α)': p['alpha'],
                'Saturation Shape': 'Gradual' if p['alpha'] == 1.5 else 'Sharp'
            })
        return pd.DataFrame(rows)


# ==========================================
# 1. PURE DATA ENGINE (48 Variables)
# ==========================================
class PureMMMDataEngine:
    def __init__(self, target_col='units_sold', date_col='week_start_date'):
        self.target_col = target_col
        self.date_col   = date_col
        self.scaler     = MinMaxScaler()
        self.adstock_engine = AdstockHillEngine()

    def process_data(self, filepath):
        print("🚀 INITIALIZING PURE DATA ENGINE (48 Variables)...")
        df = pd.read_csv(filepath)

        if self.date_col in df.columns:
            df[self.date_col] = pd.to_datetime(df[self.date_col])
            df = df.sort_values(self.date_col).reset_index(drop=True)

        y = df[self.target_col]
        X = df.drop(columns=[c for c in [self.target_col, self.date_col] if c in df.columns])

        # ── Chronological split BEFORE adstock (prevents data leakage) ──────
        split_idx = int(len(X) * 0.8)
        X_train_raw = X.iloc[:split_idx].reset_index(drop=True)
        X_test_raw  = X.iloc[split_idx:].reset_index(drop=True)
        y_train     = y.iloc[:split_idx].reset_index(drop=True)
        y_test      = y.iloc[split_idx:].reset_index(drop=True)

        # ── Apply Adstock + Hill on training data, then transform test ───────
        # FIT on train only — avoids future data leaking into transformation
        X_train_transformed = self.adstock_engine.fit_transform(X_train_raw, y_train)
        X_test_transformed  = self.adstock_engine.transform(X_test_raw)

        # ── Scale after transformation ────────────────────────────────────────
        X_train_scaled = pd.DataFrame(
            self.scaler.fit_transform(X_train_transformed),
            columns=X_train_transformed.columns
        )
        X_test_scaled = pd.DataFrame(
            self.scaler.transform(X_test_transformed),
            columns=X_test_transformed.columns
        )

        print(f"📊 Splitting Complete: {X_train_scaled.shape[1]} features ready for training.")
        print(f"   Train: {len(X_train_scaled)} weeks | Test: {len(X_test_scaled)} weeks\n")

        # Print adstock param summary
        print("📋 ADSTOCK + HILL PARAMETER SUMMARY:")
        print(self.adstock_engine.get_params_summary().to_string(index=False))
        print()

        return X_train_scaled, X_test_scaled, y_train, y_test


# ==========================================
# 2. MASTER UNIFIED TRACKER
# ==========================================
class UnifiedMMMTracker:
    def __init__(self, X_train, y_train, X_test, y_test):
        self.X_train = X_train
        self.y_train = y_train
        self.X_test  = X_test
        self.y_test  = y_test
        self.trained_models  = {}
        self.predictions     = {}
        self.feature_weights = {}

    def train_linear_models(self):
        lr = LinearRegression()
        lr.fit(self.X_train, self.y_train)
        self.trained_models['Linear Regression'] = lr
        self.predictions['Linear Regression']    = lr.predict(self.X_test)
        self.feature_weights['Linear Regression'] = lr.coef_
        print("✅ Linear Regression model finished training.")

        ridge = Ridge(alpha=50.0)
        ridge.fit(self.X_train, self.y_train)
        self.trained_models['Ridge (L2 - Smooth Distribution)'] = ridge
        self.predictions['Ridge (L2 - Smooth Distribution)']    = ridge.predict(self.X_test)
        self.feature_weights['Ridge (L2 - Smooth Distribution)'] = ridge.coef_
        print("✅ Ridge (L2 - Smooth Distribution) finished training.")

        lasso = Lasso(alpha=2.0, max_iter=10000)
        lasso.fit(self.X_train, self.y_train)
        self.trained_models['Lasso (L1 - Strict Selection)'] = lasso
        self.predictions['Lasso (L1 - Strict Selection)']    = lasso.predict(self.X_test)
        self.feature_weights['Lasso (L1 - Strict Selection)'] = lasso.coef_
        print("✅ Lasso (L1 - Strict Selection) finished training.")

    def train_ols_model(self):
        X_train_sm = sm.add_constant(self.X_train)
        X_test_sm  = sm.add_constant(self.X_test)
        ols_model  = sm.OLS(self.y_train.values, X_train_sm).fit()
        self.trained_models['OLS Model']  = ols_model
        self.predictions['OLS Model']     = ols_model.predict(X_test_sm)
        self.feature_weights['OLS Model'] = ols_model.params[1:]
        print("✅ OLS Model finished training.")

    def train_tree_models(self):
        rf = RandomForestRegressor(n_estimators=300, max_depth=10, random_state=42)
        rf.fit(self.X_train, self.y_train)
        self.trained_models['Random Forest (Bagging)'] = rf
        self.predictions['Random Forest (Bagging)']    = rf.predict(self.X_test)
        self.feature_weights['Random Forest (Bagging)'] = rf.feature_importances_
        print("✅ Random Forest (Bagging) finished training.")

        xgb = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=5, random_state=42)
        xgb.fit(self.X_train, self.y_train)
        self.trained_models['XGBoost (Boosting)'] = xgb
        self.predictions['XGBoost (Boosting)']    = xgb.predict(self.X_test)
        self.feature_weights['XGBoost (Boosting)'] = xgb.feature_importances_
        print("✅ XGBoost (Boosting) finished training.")

    def train_bayesian_model(self, media_keywords=['tv','youtube','facebook','radio','tiktok','influencer']):
        # After adstock+hill, media cols are renamed with _adstock_hill suffix
        media_cols = [col for col in self.X_train.columns
                      if any(k in col.lower() for k in media_keywords)]
        base_cols  = [col for col in self.X_train.columns if col not in media_cols]

        with pm.Model() as bayes_model:
            media_data = pm.Data('media_data', self.X_train[media_cols].values)
            base_data  = pm.Data('base_data',  self.X_train[base_cols].values)
            y_data     = pm.Data('y_data',     self.y_train.values)

            y_std = float(self.y_train.std())

            intercept  = pm.Normal('intercept', mu=float(self.y_train.mean()), sigma=y_std)
            beta_base  = pm.Normal('beta_base', mu=0, sigma=y_std, shape=len(base_cols))
            base_effect = pm.math.dot(base_data, beta_base)

            # HalfNormal enforces non-negative media effect (ads can't cause negative sales)
            beta_media  = pm.HalfNormal('beta_media', sigma=y_std, shape=len(media_cols))
            media_effect = pm.math.dot(media_data, beta_media)

            mu    = intercept + base_effect + media_effect
            sigma = pm.HalfNormal('sigma', sigma=y_std)
            pm.Normal('units_sold', mu=mu, sigma=sigma, observed=y_data)

            trace = pm.sample(draws=1000, tune=1000, progressbar=False, return_inferencedata=True)

        self.trained_models['Bayesian (PyMC)'] = bayes_model

        with bayes_model:
            pm.set_data({
                'media_data': self.X_test[media_cols].values,
                'base_data':  self.X_test[base_cols].values,
                'y_data':     self.y_test.values
            })
            post_pred = pm.sample_posterior_predictive(trace, progressbar=False)
            preds = post_pred.posterior_predictive['units_sold'].median(
                dim=['chain', 'draw']).values

        self.predictions['Bayesian (PyMC)'] = preds

        full_weights = np.zeros(len(self.X_train.columns))
        median_beta_media = trace.posterior['beta_media'].median(dim=['chain', 'draw']).values
        median_beta_base  = trace.posterior['beta_base'].median(dim=['chain', 'draw']).values

        for i, col in enumerate(self.X_train.columns):
            if col in media_cols:
                full_weights[i] = median_beta_media[media_cols.index(col)]
            else:
                full_weights[i] = median_beta_base[base_cols.index(col)]

        self.feature_weights['Bayesian (PyMC)'] = full_weights
        print("✅ Bayesian model finished training.")

    def generate_leaderboard(self):
        results = []
        for name, preds in self.predictions.items():
            mape = mean_absolute_percentage_error(self.y_test, preds) * 100
            rmse = np.sqrt(mean_squared_error(self.y_test, preds))
            r2   = r2_score(self.y_test, preds)
            results.append({
                'Model Architecture': name,
                'MAPE (%)': round(mape, 2),
                'RMSE':     round(rmse, 2),
                'R²':       round(r2, 4)
            })

        leaderboard = pd.DataFrame(results).sort_values(
            by='MAPE (%)', ascending=True).reset_index(drop=True)

        print("\n==============================================")
        print("🏆 Leaderboard")
        print("==============================================")
        print(leaderboard.to_string())
        return leaderboard

    def extract_contributions(self):
        print("\n==============================================")
        print("📊 MATHEMATICAL CONTRIBUTIONS (Cross-Model)")
        print("==============================================\n")

        categories = {
            'TV Spend':         'tv',
            'Facebook Spend':   'facebook',
            'YouTube Spend':    'youtube',
            'Radio Spend':      'radio',
            'TikTok Spend':     'tiktok',
            'Influencer Spend': 'influencer'
        }

        master_table = {}
        for model_name, weights in self.feature_weights.items():
            bucketed_pct = {k: 0.0 for k in categories.keys()}
            base_pct     = 0.0

            if 'Forest' in model_name or 'XGBoost' in model_name:
                impacts = weights * 100
            else:
                raw_impacts  = np.abs(weights * self.X_train.mean().values)
                total_impact = np.sum(raw_impacts)
                impacts = (raw_impacts / total_impact) * 100 if total_impact > 0 else raw_impacts

            for col_name, pct in zip(self.X_train.columns, impacts):
                matched = False
                for bucket_name, keyword in categories.items():
                    if keyword in col_name.lower():
                        bucketed_pct[bucket_name] += pct
                        matched = True
                        break
                if not matched:
                    base_pct += pct

            col_data = [f"{base_pct:.2f}%"] + [
                f"{bucketed_pct[k]:.2f}%" for k in categories.keys()]
            master_table[model_name] = col_data

        index_labels = ['Base Sales (Macro/Dist)'] + list(categories.keys())
        df_contributions = pd.DataFrame(master_table, index=index_labels)
        print(df_contributions.to_string())
        return df_contributions


# ==========================================
# EXECUTION BLOCK
# ==========================================
data_engine = PureMMMDataEngine()
X_train, X_test, y_train, y_test = data_engine.process_data('fmcg_synthetic_mmm_dataset_v2.csv')

tracker = UnifiedMMMTracker(X_train, y_train, X_test, y_test)
tracker.train_linear_models()
tracker.train_tree_models()
tracker.train_ols_model()
tracker.train_bayesian_model()

leaderboard_df    = tracker.generate_leaderboard()
contribution_df   = tracker.extract_contributions()

🚀 INITIALIZING PURE DATA ENGINE (48 Variables)...

📡 ADSTOCK + HILL SATURATION TRANSFORMATION
Channel                               Best λ   Best α  Corr with Sales
------------------------------------------------------------
tv_national_spend                        0.3      2.5           0.6956
radio_spend                              0.9      2.5          -0.1506
youtube_spend                            0.3      1.5           0.2981
facebook_spend                           0.3      2.5           0.3744
tiktok_spend                             0.9      2.5          -0.0978
influencer_marketing_spend               0.9      2.5          -0.2579
competitor_A_tv_spend                    0.9      2.5          -0.2120
competitor_B_tv_spend                    0.9      2.5          -0.1146
✅ 8 media channels transformed.
   Order: Geometric Adstock → Hill Saturation (γ=0.5)
   Param selection: max correlation with units_sold

📊 Splitting Complete: 48 features ready for training.
   Train: 124